# Phase 2 · Custom CNN: Architecture & Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MartinAlani/project_deep_learning_week_7/blob/main/notebooks/02_custom_cnn.ipynb)

**Goals of this notebook**
1. Rebuild the data pipeline from Phase 1 (each notebook is self-contained).
2. Design a CNN with **convolutional**, **pooling** and **fully connected** layers.
3. Train it with the **Adam** optimizer.
4. Prevent overfitting with **early stopping**, dropout and data augmentation.
5. Save the best model to Google Drive and plot the learning curves.

> 💡 **In Colab:** *Runtime → Change runtime type → T4 GPU*. Training takes about **15-25 minutes** on a T4.

## 1. Setup: imports, seed and Google Drive

Colab deletes all files when the session ends. To keep the trained model (we need it in Phase 3, 4 and for the app), we save it to **Google Drive**. Colab will ask for permission to access your Drive: accept it.

In [ ]:
import os
import json
import time

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

# Save outputs to Google Drive when running in Colab, otherwise to the local repo folder
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/project_deep_learning_week_7"
except ImportError:
    PROJECT_DIR = ".."

MODELS_DIR = os.path.join(PROJECT_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)
print("Models will be saved in:", MODELS_DIR)

## 2. Data pipeline (same as Phase 1)

We repeat the exact same steps as in Phase 1. Because we use the same `SEED`, the train/validation split is **identical**: this lets us compare all our models fairly.

In [ ]:
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

# Load and split
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train_full, y_test = y_train_full.flatten(), y_test.flatten()

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=5000, stratify=y_train_full, random_state=SEED
)

def make_dataset(x, y, training=False):
    """Turn NumPy arrays into a batched, prefetched tf.data.Dataset."""
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(buffer_size=len(x), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val)

print("Train:", x_train.shape, "| Validation:", x_val.shape)

## 3. Model architecture

We build a **VGG-style** CNN: three blocks of two convolutions + pooling, with an increasing number of filters (32 → 64 → 128), followed by fully connected layers.

```
Input 32×32×3 (raw pixels 0-255)
 ├─ Data augmentation          (active only during training)
 ├─ Rescaling 1/255            (normalization inside the model)
 ├─ Block 1: Conv(32)  ×2 → MaxPool → Dropout 0.2   → 16×16×32
 ├─ Block 2: Conv(64)  ×2 → MaxPool → Dropout 0.3   →  8×8×64
 ├─ Block 3: Conv(128) ×2 → MaxPool → Dropout 0.4   →  4×4×128
 ├─ Flatten                                          →  2048
 ├─ Dense(128) + ReLU + Dropout 0.5
 └─ Dense(10) + Softmax                              →  class probabilities
```

**Design choices**
- **3×3 kernels** with `padding="same"`: small filters stacked twice see a 5×5 area with fewer parameters (the VGG idea); `same` keeps the spatial size so only pooling shrinks the image.
- **Batch Normalization** after each convolution: keeps activations on a stable scale → faster, more stable training.
- **Increasing dropout** (0.2 → 0.5): deeper layers have more parameters and overfit more easily.
- **Softmax** output: 10 probabilities that sum to 1.

In [ ]:
def conv_block(x, filters, dropout_rate, block_id):
    """Two Conv2D → BatchNorm → ReLU layers, followed by MaxPooling and Dropout.

    Args:
        x: input tensor.
        filters: number of convolutional filters in this block.
        dropout_rate: fraction of activations dropped after pooling.
        block_id: integer used to name the layers (helps reading model.summary()).
    """
    for i in (1, 2):
        x = layers.Conv2D(filters, kernel_size=3, padding="same",
                          name=f"block{block_id}_conv{i}")(x)
        x = layers.BatchNormalization(name=f"block{block_id}_bn{i}")(x)
        x = layers.Activation("relu", name=f"block{block_id}_relu{i}")(x)
    x = layers.MaxPooling2D(pool_size=2, name=f"block{block_id}_pool")(x)
    x = layers.Dropout(dropout_rate, name=f"block{block_id}_dropout")(x)
    return x


def build_custom_cnn(input_shape=(32, 32, 3), num_classes=10):
    """Build the custom VGG-style CNN for CIFAR-10."""
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
    ], name="data_augmentation")

    inputs = keras.Input(shape=input_shape, name="image")
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255, name="rescaling")(x)

    # Feature extraction: convolution + pooling blocks
    x = conv_block(x, 32, 0.2, block_id=1)
    x = conv_block(x, 64, 0.3, block_id=2)
    x = conv_block(x, 128, 0.4, block_id=3)

    # Classification: fully connected layers
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(128, activation="relu", name="dense")(x)
    x = layers.Dropout(0.5, name="dense_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    return keras.Model(inputs, outputs, name="custom_cnn")


model = build_custom_cnn()
model.summary()

**How to read `model.summary()`**
- **Output Shape**: size of the data after each layer, e.g. `(None, 16, 16, 32)` = 16×16 image with 32 feature maps. `None` is the batch size (any number of images).
- **Param #**: number of weights the layer learns. Pooling, dropout and activation layers have **0**: they don't learn anything.
- **Total params**: about 0.55 million, a small model by modern standards (VGG16 has 138 million).

A diagram of the architecture (useful for the report):

In [ ]:
try:
    keras.utils.plot_model(model, to_file="custom_cnn_architecture.png",
                           show_shapes=True, show_layer_names=True, dpi=70)
    from IPython.display import Image, display
    display(Image("custom_cnn_architecture.png"))
except Exception as e:  # plot_model needs pydot + graphviz (pre-installed in Colab)
    print("Could not draw the diagram:", e)

## 4. Compile the model

- **Optimizer: Adam** (learning rate 0.001). Adam adapts the step size for every weight automatically, so it usually converges faster and needs less tuning than plain SGD.
- **Loss: sparse categorical cross-entropy**. It measures how far the predicted probabilities are from the true class. *Sparse* means our labels are integers (0-9) instead of one-hot vectors.
- **Metric: accuracy**, the percentage of correctly classified images.

In [ ]:
LEARNING_RATE = 1e-3

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## 5. Callbacks: early stopping & friends

Callbacks are functions Keras runs at the end of every epoch.

| Callback | What it does |
|---|---|
| **EarlyStopping** | Stops training when `val_loss` hasn't improved for 10 epochs, and **restores the best weights**. Prevents overfitting and saves time. |
| **ReduceLROnPlateau** | If `val_loss` stalls for 4 epochs, halves the learning rate. Smaller steps help the model fine-tune once it is close to a good solution. |
| **ModelCheckpoint** | Saves the best model (lowest `val_loss`) to Google Drive, so it survives if Colab disconnects. |

We monitor **validation** loss, never test data.

In [ ]:
EPOCHS = 100   # maximum; early stopping will usually stop much earlier
MODEL_PATH = os.path.join(MODELS_DIR, "custom_cnn.keras")

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4,
                                      min_lr=1e-5, verbose=1),
    keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss",
                                    save_best_only=True, verbose=0),
]

## 6. Train 🚀

Each line printed is one **epoch** (one full pass over the 45,000 training images). Watch:
- `accuracy` / `loss` → performance on **train** data (with augmentation)
- `val_accuracy` / `val_loss` → performance on **validation** data (images the model never trains on)

If `accuracy` keeps rising while `val_accuracy` stops improving, the model is starting to overfit, and early stopping will step in.

In [ ]:
start = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

training_minutes = (time.time() - start) / 60
print(f"\nTraining time: {training_minutes:.1f} minutes")
print(f"Epochs run: {len(history.history['loss'])}")

## 7. Learning curves

These plots are the best way to understand **how** the model learned, and they are a key figure for the report.

In [ ]:
def plot_history(history, title="Custom CNN"):
    """Plot training vs validation accuracy and loss for each epoch."""
    hist = history.history
    epochs = range(1, len(hist["loss"]) + 1)
    best_epoch = int(np.argmin(hist["val_loss"])) + 1

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(epochs, hist[metric], label=f"train {metric}")
        ax.plot(epochs, hist[f"val_{metric}"], label=f"validation {metric}")
        ax.axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch ({best_epoch})")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric.capitalize())
        ax.set_title(f"{title} · {metric}")
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR, "custom_cnn_learning_curves.png"), dpi=120)
    plt.show()

plot_history(history)

**How to interpret the curves**
- Both curves going up (accuracy) / down (loss) together → the model is **learning**.
- Train much better than validation, with validation getting worse → **overfitting**.
- Here, train accuracy may even be *lower* than validation accuracy: this is normal, because train images are **augmented** (harder) and dropout is active during training only.

## 8. Quick check on the validation set

The full evaluation (precision, recall, F1, confusion matrix and the **test set**) is done in Phase 3. Here we just confirm the best weights were restored.

In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"Validation accuracy: {val_acc:.4f}")
print(f"Validation loss:     {val_loss:.4f}")

## 9. Save training history & settings

We save the history and hyper-parameters to a JSON file so we can redraw the curves and compare models in the report without retraining.

In [ ]:
results = {
    "model": "custom_cnn",
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": EPOCHS,
    "epochs_run": len(history.history["loss"]),
    "training_minutes": round(training_minutes, 1),
    "total_params": int(model.count_params()),
    "val_accuracy": float(val_acc),
    "val_loss": float(val_loss),
    "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
}

with open(os.path.join(MODELS_DIR, "custom_cnn_history.json"), "w") as f:
    json.dump(results, f, indent=2)

print("Saved files in", MODELS_DIR)
for name in sorted(os.listdir(MODELS_DIR)):
    print(" -", name)

## ✅ Phase 2 summary

| Item | Value |
|---|---|
| Architecture | VGG-style CNN: 3 conv blocks (32-64-128 filters) + Dense(128) + Softmax |
| Regularization | Data augmentation, Batch Normalization, Dropout (0.2 → 0.5), early stopping |
| Optimizer | Adam, initial learning rate 0.001, halved on plateau |
| Batch size | 64 |
| Epochs | up to 100, early stopping with patience 10 |
| Saved model | `models/custom_cnn.keras` in Google Drive |

👉 **For the report:** write down the validation accuracy, number of epochs and training time, and save the architecture diagram and the learning curves.

**Next step → Phase 3:** full evaluation on the test set (accuracy, precision, recall, F1-score, confusion matrix).